In [15]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from pybamm import exp
from pybamm import tanh

Defining Model Variables

In [16]:

xi = pybamm.SpatialVariable(
    "xi", domain="SEI layer", coord_sys="cartesian")

CL_atom = pybamm.Variable(
    "concentration of netutral lithium atoms in the SEI [mol.m-3]",  domain="SEI layer")
CL_ion = pybamm.Variable(
    "concentration of llithium ions in the SEI [mol.m-3]",  domain="SEI layer")
Phi_SEI = pybamm.Variable("Potential in the SEI [V]",  domain="SEI layer")
L_SEI = pybamm.Variable("Thickness of SEI [m]")

In [17]:
model = pybamm.lithium_ion.BaseModel()

Defining parameters of the model

In [18]:
T = pybamm.Parameter('Initial temperature [K]')
R = pybamm.Parameter('Ideal gas constant [J.K-1.mol-1]')
F = pybamm.Parameter("Faraday constant [C.mol-1]")
D_Li_atom = pybamm.Parameter(
    'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]')
D_Li_ion = pybamm.Parameter(
    'diffusion coefficient of llithium ions in the SEI [m2.s-1]')

CL_atom_0 = pybamm.Parameter(
    "initial concentration of netutral lithium atoms in the SEI [mol.m-3]")
CL_ion_0 = pybamm.Parameter(
    "initial concentration of llithium ions in the SEI [mol.m-3]")

Phi_SEI_0 = pybamm.Parameter("initial potential in the SEI [V]")


M_SEI = pybamm.Parameter("molar weight of SEI material [kg.mol-1]")
rho_SEI = pybamm.Parameter("density of SEI material [kg.m-3]")
n_SEI = pybamm.Parameter("electrone number [-]")


L_tun = pybamm.Parameter("length of the tunneling [m]")
L_SEI_0 = pybamm.Parameter("initial thickness of SEI [m]")

CL_ion_max = pybamm.Parameter(
    "Maximum concentration in negative electrode [mol.m-3]")

CL_ion_e = pybamm.Parameter(
    "concentration of llithium ions in the electrolyte [mol.m-3]")

# j0_0 = pybamm.Parameter("Negative electrode exchange-current density [A.m-2]")

J_total = pybamm.Parameter("Total current density [A.m-2]")

j_sei_0 = pybamm.Parameter("SEI reaction exchange current density [A.m-2]")

a_n = pybamm.Parameter("Surface area per unit volume [m-1]")






In [19]:
param = pybamm.ParameterValues(
    {
        'Initial temperature [K]': 298.15,
        'Ideal gas constant [J.K-1.mol-1]': 8.314462618,
        "Faraday constant [C.mol-1]": 96485.33212,
        'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]': 1e-14,
        'diffusion coefficient of llithium ions in the SEI [m2.s-1]': 1e-14,
        'initial concentration of netutral lithium atoms in the SEI [mol.m-3]': 1000,
        'initial concentration of llithium ions in the SEI [mol.m-3]': 1000,
        'initial potential in the SEI [V]': 2.5,
        'molar weight of SEI material [kg.mol-1]': 0.162,
        'density of SEI material [kg.m-3]': 1690,
        'electrone number [-]': 1,
        'length of the tunneling [m]': 5e-9,
        'initial thickness of SEI [m]': 1e-9,
        'Maximum concentration in negative electrode [mol.m-3]': 33000,
        "Total current density [A.m-2]": 2,
        "SEI reaction exchange current density [A.m-2]": 1.5e-07,  
        "concentration of llithium ions in the electrolyte [mol.m-3]": 1000,
        "Surface area per unit volume [m-1]": 487864,


    }
)

In [20]:
# Unclear how to define the source terms
A = 10
R_CLi_ions =  - A*CL_ion*pybamm.exp(-xi*L_SEI/L_tun)
R_CLi_atom = -  R_CLi_ions

NL_ions = - D_Li_ion / L_SEI * pybamm.grad(CL_ion)
# define the flux for netutral lithium atoms
NL_atom = - D_Li_atom / L_SEI * \
    pybamm.grad(CL_atom)

Ne = - D_Li_atom  * pybamm.grad(CL_ion) \
    + D_Li_atom * F / (R*T) * CL_atom * pybamm.grad(Phi_SEI)
V_SEI = M_SEI/(n_SEI*rho_SEI)


# J_Li_0 = NL_atom - D_Li_atom*F/(R*T * L_SEI)*CL_atom*pybamm.grad(Phi_SEI)
# J_tun = A * J_Li_0 * pybamm.exp(- L_SEI / L_tun)
# Je = J_Li_0 + J_tun
V= 3
eta_sei = pybamm.BoundaryValue(Phi_SEI, "right")
alpha = 0.5
J_sei = - j_sei_0 * pybamm.exp(-alpha*F/(R*T)*eta_sei)
# define the rhs equation
dL_SEI_dt = - V_SEI / F * J_sei
dCL_ion_dt = dL_SEI_dt / L_SEI * pybamm.inner(
    xi, pybamm.grad(CL_ion)) - 1 / L_SEI * pybamm.div(NL_ions) + R_CLi_ions
dCL_atom_dt = dL_SEI_dt / L_SEI * pybamm.inner(
    xi, pybamm.grad(CL_atom)) - 1 / L_SEI * pybamm.div(NL_atom) + R_CLi_atom

In [21]:

model.algebraic = {Phi_SEI: pybamm.div(Ne)-L_SEI **2 * R_CLi_atom }
model.rhs = {CL_ion: dCL_ion_dt, CL_atom: dCL_atom_dt, L_SEI: dL_SEI_dt}

In [22]:
model.variables = {
    "concentration of llithium ions in the SEI [mol.m-3]": CL_ion,
    "concentration of netutral lithium atoms in the SEI [mol.m-3]": CL_atom,
    "Potential in the SEI [V]": Phi_SEI,
    "Thickness of SEI [m]": L_SEI,
    "J_sei": J_sei,
}

In [23]:
model.initial_conditions = {CL_ion: CL_ion_0,
                            Phi_SEI:  0, L_SEI: L_SEI_0, CL_atom: CL_atom_0}
phi_Lsei = pybamm.BoundaryValue(Phi_SEI, "right")
# lbc_CL_ion = pybamm.BoundaryValue(L_SEI * (J_total /F - A * CL_ion * L_tun +\
#         D_Li_atom * F / (R*T) * CL_atom * (phi_Lsei-V)/L_SEI**2 *0 ), "left")
lbc_CL_ion = pybamm.BoundaryValue( (L_SEI * J_total /F -(F/(R*T)) * D_Li_atom * CL_atom * pybamm.grad(Phi_SEI) ), "left")
rbc_CL_ion = CL_ion_e

lbc_CL_atom = 0
rbc_CL_atom = - pybamm.BoundaryValue(L_SEI * J_sei/(F*D_Li_atom), "right")
# rbc_CL_atom = 0



lbc_phi_SEI = V
rbc_phi_SEI = -pybamm.BoundaryValue(2* R * T * L_SEI * J_sei/(D_Li_atom* F**2 * CL_atom), "right")

model.boundary_conditions = {
    CL_ion: {"left": (lbc_CL_ion, "Neumann"), 
            "right": (rbc_CL_ion, "Dirichlet")},
    CL_atom: {"left": (lbc_CL_atom, "Neumann"),
             "right": (rbc_CL_atom, "Neumann")},
    Phi_SEI: {"left": (lbc_phi_SEI, "Dirichlet"),
              "right": (rbc_phi_SEI, "Neumann")},
}

In [24]:

geometry = pybamm.Geometry(
    {"SEI layer": {xi: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(1)}}})

In [25]:
param.process_model(model)
param.process_geometry(geometry)
submesh_types = {"SEI layer": pybamm.Uniform1DSubMesh}
var_pts = {xi: 20}

mesh = pybamm.Mesh(geometry, submesh_types, var_pts)
spatial_methods = {"SEI layer": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)
disc.process_model(model)

ValueError: Can't take the boundary value of a symbol that evaluates on edges

In [ ]:
# solver = pybamm.ScipySolver()
# pybamm.CasadiSolver(mode="fast")
# pybamm.settings.max_num_steps = 1000000000

# solver = pybamm.IDAKLUSolver(extra_options_setup={"max_num_steps": 100000})
solver = pybamm.IDAKLUSolver()

In [ ]:
sim = pybamm.Simulation(
    model,
    geometry=geometry,
    parameter_values=param,
    var_pts=var_pts,
    spatial_methods=spatial_methods,
    solver=solver,
)
sol = sim.solve([0, 1e-7])

In [ ]:

pybamm.dynamic_plot(sol, output_variables=["concentration of llithium ions in the SEI [mol.m-3]",
                                           "concentration of netutral lithium atoms in the SEI [mol.m-3]",
                                           "Potential in the SEI [V]",
                                           "Thickness of SEI [m]"])

interactive(children=(FloatSlider(value=0.0, description='t', max=1e-07, step=9.999999999999999e-10), Output()…